In [3]:
import pandas as pd
train = pd.read_csv("../data/processed/train.csv")

print(train["Mileage"].value_counts().sort_index().head(40))
print(train["Year"].value_counts().sort_index().head(10))
print(train["Fiscal Power"].value_counts().head(10))
print(train["Equipment"].dropna().head(5).tolist())
print(train[["Condition", "Gearbox", "Fuel", "Origin", "First Owner"]].nunique())

Mileage
0 - 4 999            2549
10 000 - 14 999      1416
100 000 - 109 999    2616
110 000 - 119 999    1871
120 000 - 129 999    2269
130 000 - 139 999    2138
140 000 - 149 999    2207
15 000 - 19 999      1450
150 000 - 159 999    2352
160 000 - 169 999    2057
170 000 - 179 999    2078
180 000 - 189 999    2123
190 000 - 199 999    2893
20 000 - 24 999      1697
200 000 - 249 999    6523
25 000 - 29 999      1319
250 000 - 299 999    3340
30 000 - 34 999      1268
300 000 - 349 999    1998
35 000 - 39 999       833
350 000 - 399 999     750
40 000 - 44 999       827
400 000 - 449 999     558
45 000 - 49 999       694
450 000 - 499 999     299
5 000 - 9 999        1538
50 000 - 54 999       763
55 000 - 59 999       608
60 000 - 64 999       905
65 000 - 69 999       631
70 000 - 74 999       861
75 000 - 79 999       720
80 000 - 84 999      1133
85 000 - 89 999       790
90 000 - 94 999      1166
95 000 - 99 999      1229
Plus de 500 000       535
Name: count, dtype: int64
Year

In [4]:
import sys; sys.path.append("..")
from src.features import add_features
train = add_features(train)
train[["Mileage", "mileage_num", "Year", "car_age", "Fiscal Power", "fiscal_power_num"]].sample(10, random_state=42)

,Mileage,mileage_num,Year,car_age,Fiscal Power,fiscal_power_num
38499,250 000 - 299 999,274999.5,2014,10.0,8 CV,8.0
4772,10 000 - 14 999,12499.5,1980 ou plus ancien,44.0,8 CV,8.0
1182,90 000 - 94 999,92499.5,2019,5.0,6 CV,6.0
38772,95 000 - 99 999,97499.5,2019,5.0,6 CV,6.0
4935,200 000 - 249 999,224999.5,2010,14.0,8 CV,8.0
17724,110 000 - 119 999,114999.5,2016,8.0,6 CV,6.0
57318,0 - 4 999,2499.5,1985,39.0,7 CV,7.0
54332,95 000 - 99 999,97499.5,2021,3.0,6 CV,6.0
44897,190 000 - 199 999,194999.5,2011,13.0,8 CV,8.0
28058,200 000 - 249 999,224999.5,1997,27.0,11 CV,11.0


In [5]:
print(train["Number of Doors"].dtype)
print(train["Number of Doors"].value_counts(dropna=False).head(10))

float64
Number of Doors
5.0    49945
NaN     5729
3.0     3330
Name: count, dtype: int64


In [6]:
num_cols = ["mileage_num", "car_age", "fiscal_power_num", "Number of Doors"]  # ou "doors_num"
cat_cols = ["Brand", "Model", "Condition", "Gearbox", "Fuel", "Origin",
            "First Owner", "Location", "Sector"]

manquantes = [c for c in num_cols + cat_cols if c not in train.columns]
print("Colonnes introuvables :", manquantes)   # attendu : []

Colonnes introuvables : []


In [7]:
X_train = train.drop(columns="Price")
y_train = train["Price"]
print(X_train.shape, y_train.shape)

(59004, 18) (59004,)


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

In [9]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

dummy = Pipeline([("prep", prep), ("reg", DummyRegressor(strategy="median"))])
baseline = Pipeline([("prep", prep), ("reg", LinearRegression())])

mae_dummy = -cross_val_score(dummy, X_train, y_train, cv=5,
                             scoring="neg_mean_absolute_error").mean()
print("MAE dummy (prédit toujours la médiane) :", round(mae_dummy))

MAE dummy (prédit toujours la médiane) : 74979


In [10]:
scores = cross_val_score(baseline, X_train, y_train, cv=5,
                         scoring="neg_mean_absolute_error", n_jobs=-1)
print("MAE par pli :", (-scores).round())
print("MAE baseline (CV) :", round(-scores.mean()))

MAE par pli : [45143. 45419. 44929. 44500. 44536.]
MAE baseline (CV) : 44905


## Résultats de la baseline (validation croisée à 5 plis sur le train)

| Modèle | MAE (CV, MAD) |
|---|---|
| Dummy (prédit toujours la médiane du train) | 74 979 |
| Régression linéaire (pipeline complet) | 44 905 |

**Lecture :**
- La régression linéaire réduit l'erreur d'environ 40 % par rapport au modèle « médiane ».
- Les 5 plis donnent 44 500 à 45 419 : le résultat est stable.
- Sur une voiture à 110 000 MAD (la médiane), une erreur de 44 905 MAD reste importante :
  c'est une référence à battre, pas un score satisfaisant.
- Unité : MAD (hypothèse de travail, non confirmée par la description du dataset).

**Pipeline :** conversions déterministes (`src/features.py`) ; ensuite, dans le pipeline,
imputation médiane (numériques) et valeur la plus fréquente (catégories), puis `OneHotEncoder`.
Toutes les statistiques sont apprises sur les plis d'entraînement uniquement.
`test.csv` n'a pas été ouvert.

**Pas encore fait :** `Equipment` (non utilisé), écrêtage statistique de la cible
(à décider après l'EDA), modèles non linéaires.